In [11]:
from pathlib import Path

import numpy
from astropy.io import fits
from matplotlib import pyplot
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pandas import read_csv

ALL_DATASETS_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid")
RESULTS_SEXTRACTOR_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR")

In [12]:
VERSION = "2.0.1_sextractor"

DATASET_DIR = ALL_DATASETS_DIR / VERSION
IMAGES_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "IMAGES"
VIGNET_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "VIGNET"
RESULTS_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "RESULTS"
PLOTS_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# functions

In [13]:
def print_single_fits(name, title, directory, divide=1):
    fig = pyplot.figure()
    img = fits.open(f"{name}.fits")[0].data
    pyplot.title(f"{title} => Min: {numpy.min(img):.3f}, Max: {numpy.max(img):.3f}")
    pyplot.imshow(img, cmap="gray", vmax=numpy.max(img) / divide)
    pyplot.colorbar()
    fig.show()
    fig.savefig(directory, dpi=300)


def plot_vignettes(vignettes_images, pos_x, pos_y, directory):
    fig, axes = pyplot.subplots(3, 4, figsize=(12, 9))
    axes = axes.flatten()

    for i, (fits_file, x, y) in enumerate(zip(vignettes_images[:12], pos_x, pos_y)):
        image = fits.open(fits_file)[0].data
        img = axes[i].imshow(image, cmap="gray")
        axes[i].set_title(f"{fits_file.stem} {x:.2f} {y:.2f}")
        divider = make_axes_locatable(axes[i])
        cax1 = divider.append_axes("right", size="5%", pad=0.05)
        pyplot.colorbar(img, cax=cax1)
        axes[i].axis("off")

    for j in range(len(vignettes_images), 12):
        axes[j].axis("off")

    fig.tight_layout()
    fig.show()
    fig.savefig(directory / "vignettes.pdf", dpi=300)


def print_fits(names, titles, directory, divide=1):
    fig, axes = pyplot.subplots(3, 2, figsize=(12, 12))
    fig.suptitle("FITS Image Grid", fontsize=16)

    for i, ax in enumerate(axes.flat):
        if i < len(names):
            img = fits.open(f"{names[i]}.fits")[0].data
            if i == 0 or i == 1:
                img = numpy.arcsin(img)
            ax.set_title(f"{titles[i]} => Min: {numpy.min(img):.3f}, Max: {numpy.max(img):.3f}")
            im = ax.imshow(img, cmap="gray", vmax=numpy.max(img) / divide)
            fig.colorbar(im, ax=ax)
        else:
            ax.axis("off")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.show()
    fig.savefig(directory / "output_sextractor.pdf", dpi=300)


def check_psf(file, directory):
    img_psf = fits.open(file)[0].data
    fig, axes = pyplot.subplots(1, 2, figsize=(12, 6))
    im1 = axes[0].imshow(numpy.log(img_psf), cmap="gray")
    axes[0].set_title("Airy Logarithmic Scale")
    divider = make_axes_locatable(axes[0])
    cax1 = divider.append_axes("right", size="5%", pad=0.05)
    pyplot.colorbar(im1, cax=cax1)
    im2 = axes[1].imshow(img_psf, cmap="gray")
    axes[1].set_title("Airy Linear Scale")
    divider = make_axes_locatable(axes[1])
    cax2 = divider.append_axes("right", size="5%", pad=0.05)
    pyplot.colorbar(im2, cax=cax2)
    pyplot.show()
    fig.savefig(directory / "psf.pdf", dpi=300)


def load_and_slice_csv(file_path: Path, nb_total: int):
    return numpy.loadtxt(file_path, delimiter=",", skiprows=1)[:nb_total].T


def res_sextractor(true_e1, pred_e1, true_e2, pred_e2, shears_g1, shears_g2, directory):
    s = 10
    alpha = 0.2
    c1, c2 = "C0", "C1"
    symbol = "epsilon"

    res_pred_g1 = pred_e1 - shears_g1
    res_pred_g2 = pred_e2 - shears_g2
    poly_g1 = numpy.polyfit(shears_g1, res_pred_g1, 1)
    poly_g2 = numpy.polyfit(shears_g2, res_pred_g2, 1)
    t = numpy.linspace(shears_g1.min(), shears_g1.max())

    fig, axes = pyplot.subplots(1, 2, figsize=(15, 5))
    axes[0].scatter(
        true_e1, pred_e1 - true_e1, alpha=alpha, color=c1, marker="x", s=s, label="avg_e1"
    )
    axes[0].scatter(
        true_e2, pred_e2 - true_e2, alpha=alpha, color=c2, marker="o", s=s, label="avg_e2"
    )
    axes[0].set_xlabel(rf"$\{symbol}^{{true}}$")
    axes[0].set_ylabel(rf"$\{symbol}^{{obs}}$ - $\{symbol}^{{true}}$", rotation=0, labelpad=20)
    axes[0].legend()
    axes[0].set_title("with ellips")

    label_1 = rf"$\gamma_1$ : m= {poly_g1[0]:.3f}, c={poly_g1[1]:.3f}"
    # axes[1].scatter(shears_g1, res_pred_g1, alpha=alpha, color=c1, marker="x", s=s, label=label_1)
    axes[1].scatter(shears_g1, pred_e1, alpha=alpha, color=c1, marker="x", s=s, label=label_1)
    label_2 = rf"$\gamma_2$ : m= {poly_g2[0]:.3f}, c={poly_g2[1]:.3f}"
    # axes[1].scatter(shears_g2, res_pred_g2, alpha=alpha, color=c2, marker="o", s=s, label=label_2)
    axes[1].scatter(shears_g2, pred_e2, alpha=alpha, color=c2, marker="o", s=s, label=label_2)
    pyplot.plot(t, poly_g1[1] + poly_g1[0] * t, color=c1)
    pyplot.plot(t, poly_g2[1] + poly_g2[0] * t, color=c2)
    axes[1].set_xlabel(r"$\gamma^{true}$")
    axes[1].set_ylabel(rf"$\{symbol}^{{obs}}$ - $\gamma^{{true}}$", rotation=0, labelpad=20)
    axes[1].legend()
    axes[1].set_title("with shears")

    fig.tight_layout()
    fig.savefig(directory / "results_sextractor.pdf", dpi=300)


# psf

In [ ]:
check_psf(ALL_DATASETS_DIR / "PSF" / "airy.psf.fits", PLOTS_DIR)

# images

In [ ]:
CHOICE = "simu_00002"

print_fits(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
    ],
    ["image", "fitting", "image", "fitting", "residual", "segmentation"],
    PLOTS_DIR,
)

In [ ]:
print_single_fits(IMAGES_DIR / f"background_{CHOICE}", "background", PLOTS_DIR / "background")

# vignettes

In [ ]:
CHOICE = "simu_00001"

X = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["x"]
Y = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["y"]
VIGNETTES = sorted(list((VIGNET_DIR / CHOICE).glob("*.fits")))

plot_vignettes(VIGNETTES, X, Y, PLOTS_DIR)

# results

In [18]:
RESULTS = sorted(list((RESULTS_DIR).glob("simu_*.csv")))
file_path = RESULTS_DIR / "pred_avg_ellipticies.csv"

if file_path.exists():
    data = numpy.loadtxt(file_path, delimiter=",", skiprows=1)
    pred_avg_e1, pred_avg_e2 = data[:, 0], data[:, 1]
else:
    avg_e1, avg_e2 = [], []
    for sample in RESULTS:
        e1 = read_csv(sample)["e1"]
        e2 = read_csv(sample)["e2"]
        avg_e1.append(e1.mean())
        avg_e2.append(e2.mean())
    numpy.savetxt(
        file_path, numpy.column_stack((avg_e1, avg_e2)), delimiter=",", header="avg_e1,avg_e2"
    )

In [19]:
NB_TOTAL = pred_avg_e1.shape[0]
true_avg_e1, true_avg_e2 = load_and_slice_csv(DATASET_DIR / "true_avg_ellipticies.csv", NB_TOTAL)
true_shears_g1, true_shears_g2 = load_and_slice_csv(DATASET_DIR / "true_avg_shears.csv", NB_TOTAL)

In [ ]:
res_sextractor(
    true_avg_e1, pred_avg_e1, true_avg_e2, pred_avg_e2, true_shears_g1, true_shears_g2, PLOTS_DIR
)